In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env", override=True)

db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME", "postgres")
db_user = os.getenv("DB_USER", "postgres")
db_password = os.getenv("DB_PASSWORD", "postgres")

connection_string = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
conn = create_engine(connection_string)

## Overall test / assertion / generalization exclusions

In [ ]:
import pandas as pd
import re

query = f"SELECT * FROM mv_exclusions_all"
df = pd.read_sql_query(query, conn)

# Adapt the variant names.
def format_variant(variant):
    return re.sub(r'_(\d+)_TRIES$', r'$_{\1}$', variant)

df['variant'] = df['variant'].apply(format_variant)

# Map 'level' to 'Type'.
level_map = {
    '1-TEST': 'Test',
    '2-ASSERTION': 'Assertion',
    '3-GENERALIZATION': 'Generalization'
}
df['Type'] = df['level'].map(level_map)

df = df[['variant', 'Type', 'is_included', 'excluded_by', 'count']]

display(df)

# Get unique (variant, level) pairs in original order.
ordered_pairs = df[['variant', 'Type']].drop_duplicates()

# Calculate included and excluded counts.
included = df[df['is_included'] == True].groupby(['variant', 'Type'])['count'].sum().reset_index()
excluded = df[df['is_included'] == False].groupby(['variant', 'Type'])['count'].sum().reset_index()

included = included.rename(columns={'count': 'included_count'})
excluded = excluded.rename(columns={'count': 'excluded_count'})

# Merge counts.
result = pd.merge(ordered_pairs, included, on=['variant', 'Type'], how='left')
result = pd.merge(result, excluded, on=['variant', 'Type'], how='left')

# Fill NaNs with 0 and ensure integer type.
result['included_count'] = result['included_count'].fillna(0).astype(int)
result['excluded_count'] = result['excluded_count'].fillna(0).astype(int)

# Compute Total column
result['Total'] = result['included_count'] + result['excluded_count']

result['included_pct'] = (result['included_count'] / result['Total'] * 100).round(1)
result['excluded_pct'] = (result['excluded_count'] / result['Total'] * 100).round(1)

# Format percentages
def format_pct(val):
    return ('\\phantom{0}' if val < 10 else '') + f"{val:.1f}"

result['included_pct_str'] = result['included_pct'].apply(format_pct)
result['excluded_pct_str'] = result['excluded_pct'].apply(format_pct)

# Format Included and Excluded columns as "count (pct%)"
result['Included'] = result.apply(lambda row: f"{row['included_count']}\\; ({row['included_pct_str']}\\%)", axis=1)
result['Excluded'] = result.apply(lambda row: f"{row['excluded_count']}\\; ({row['excluded_pct_str']}\\%)", axis=1)

display(result[['variant', 'Type', 'Total', 'Included', 'Excluded']])

# Build LaTeX table
lines = []
lines.append(r"\begin{table}[H]")
lines.append(r"  \caption{Included and excluded counts by variant and level.}")
lines.append(r"  \label{tab:exclusions-summary}")
lines.append(r"  \begin{tabular}{llrrr}")
lines.append(r"    \toprule")
lines.append(r"    Variant & Type & Total & \multicolumn{1}{c}{Included} & \multicolumn{1}{c}{Excluded} \\")
lines.append(r"    \midrule")

for _, row in result.iterrows():
    row_str = ' & '.join(str(row[col]) for col in ['variant', 'Type', 'Total', 'Included', 'Excluded']) + r' \\'
    lines.append(f"    {row_str}")

lines.append(r"    \bottomrule")
lines.append(r"  \end{tabular}")
lines.append(r"\end{table}")

latex_table = "\n".join(lines)
print(latex_table)

## Filtering-based test / assertion / generalization exclusions

In [ ]:
import pandas as pd
import re

# Query and DataFrame creation
query = "SELECT * FROM mv_exclusions_filtering WHERE reject > 0"
df = pd.read_sql_query(query, conn)

# Ensure integer columns are int type
df['total'] = df['total'].astype(int)
df['accept'] = df['accept'].astype(int)
df['reject'] = df['reject'].astype(int)

# Map 'level' to 'Type'
level_map = {
    '1-TEST': 'Test',
    '2-ASSERTION': 'Assertion',
    '3-GENERALIZATION': 'Generalization'
}
df['Type'] = df['level'].map(level_map)

# Format 'variant' for LaTeX subscripts
def format_variant(variant):
    return re.sub(r'_(\d+)_TRIES$', r'$_{\1}$', variant)

df['Variant'] = df['variant'].apply(format_variant)

# Add 'Defer' to 'Reject' and drop 'Defer'
df['reject'] = df['reject'] + df['defer']
df = df.drop(columns=['defer'])

# Remove 'Filter' suffix from filter names
df['filter_name'] = df['filter_name'].str.replace(r'Filter$', '', regex=True)

# Calculate percentages
df['accept_pct'] = (df['accept'] / df['total'] * 100).round(1)
df['reject_pct'] = (df['reject'] / df['total'] * 100).round(1)

def format_count_pct(count, pct):
    return f"{count}\\; ({'\\phantom{0}' if pct < 10 else ''}{pct:.1f}\\%)"

df['Accept'] = df.apply(lambda row: format_count_pct(row['accept'], row['accept_pct']), axis=1)
df['Reject'] = df.apply(lambda row: format_count_pct(row['reject'], row['reject_pct']), axis=1)

display(df[['Variant', 'Type', 'filter_name', 'total', 'Accept', 'Reject']])

# Build LaTeX table
lines = []
lines.append(r"\begin{table}[H]")
lines.append(r"  \caption{Filtering results for tests, assertions, and generalizations by filter and (generalization) variant.}")
lines.append(r"  \label{tab:exclusions-filtering}")
lines.append(r"  \begin{tabular}{lllrrr}")
lines.append(r"    \toprule")
lines.append(r"    Variant & Type & Filter Name & Total & \multicolumn{1}{c}{Accept} & \multicolumn{1}{c}{Reject} \\")  # Hardcoded headers
lines.append(r"    \midrule")

prev_type = df.iloc[0]['Type']
for i, row in df.iterrows():
    # Insert \midrule when Type changes (but not before the first group)
    if i > 0 and row['Type'] != prev_type:
        lines.append(r"    \midrule")
    prev_type = row['Type']
    row_str = ' & '.join(str(row[col]) for col in ['Variant', 'Type', 'filter_name', 'total', 'Accept', 'Reject']) + r' \\'
    lines.append(f"    {row_str}")

lines.append(r"    \bottomrule")
lines.append(r"  \end{tabular}")
lines.append(r"\end{table}")

latex_table = "\n".join(lines)
print(latex_table)

## Exclusions caused by SPF execution failures

In [ ]:
import pandas as pd

query = f"SELECT * FROM mv_exclusions_jpf"
df = pd.read_sql_query(query, conn)

category_map = {
    "ArithmeticException: div by 0": "SPF exception",
    "NoSuchMethodException": "SPF exception",
    "AssertionFailedError": "SPF exception",
    "RuntimeException: symbolic array length": "SPF exception",
    "NoUncaughtExceptionsProperty": "SPF exception",
    "ArrayIndexOutOfBoundsException (setDoubleValue)": "SPF exception",
    "ArrayIndexOutOfBoundsException (simple)": "SPF exception",
    "NullPointerException (queueMark)": "SPF exception",
    "ArrayIndexOutOfBoundsException (setLongValue)": "SPF exception",
    "NullPointerException (writeSpecificationFiles:147)": "Teralizer exception",
    "NullPointerException (writeSpecificationFiles:124)": "Teralizer exception",
    "Failed to collect specification": "Teralizer exception",
    "OutOfMemoryError: Java heap space": "OutOfMemoryError",
    "OutOfMemoryError: GC overhead": "OutOfMemoryError",
    # The following are left unchanged:
    # "PC size limit exceeded"
    # "Depth limit exceeded"
    # "Execution timeout"
}

# Apply the mapping, keeping unmapped categories as is
df['merged_category'] = df['error_category'].map(category_map).fillna(df['error_category'])

display(df[['error_category', 'merged_category', 'count']])

# Group by merged_category and sum the counts
df_merged = df.groupby('merged_category', as_index=False)['count'].sum()

# Sort by 'count' in descending order
df_merged = df_merged.sort_values(by='count', ascending=False).reset_index(drop=True)

# Add percent column
total = df_merged['count'].sum()
df_merged['percent'] = (df_merged['count'] / total * 100).round(2)

# Rename columns
df_merged.rename(columns={'merged_category': 'Error Type', 'count': 'Total', 'percent': 'Percent'}, inplace=True)

display(df_merged)

# Generate LaTeX code:
latex_table = [
    r"\begin{table}[H]",
    r"  \centering",
    r"  \caption{Number of SPF execution failures by error type.}",
    r"  \label{tab:exclusions-spf}",
    f"  \\begin{{tabular}}{{{'l' + 'r' * (len(df_merged.columns) - 1)}}}",
    r"    \toprule",
    "    " + " & ".join(df_merged.columns) + r" \\",
    r"    \midrule"
]

for _, row in df_merged.iterrows():
    values = [
        str(row[col]) if df_merged[col].dtype == "object"
        else f"{row[col]:.2f}" if isinstance(row[col], float)
        else str(row[col])
        for col in df_merged.columns
    ]
    latex_table.append("    " + " & ".join(values) + r" \\")

latex_table += [
    r"    \bottomrule",
    r"  \end{tabular}",
    r"\end{table}"
]

print("\n".join(latex_table))

## Exclusions caused by test failures

In [ ]:
import re

query = f"SELECT * FROM mv_exclusions_test_fails"
df = pd.read_sql_query(query, conn)

# Pivot as before
ordered_variants = (
    df[['variant', 'variant_order']]
    .drop_duplicates()
    .sort_values('variant_order')
    ['variant']
    .tolist()
)
df_agg = df.groupby(['failure_type', 'variant'], as_index=False)['count'].sum()
pivoted_df = df_agg.pivot(index='failure_type', columns='variant', values='count')
pivoted_df = pivoted_df[ordered_variants].fillna(0).astype(int)

display(pivoted_df)

# --- AUTOMATED HEADER GENERATION ---

# Extract base variant and tries
variant_info = []
for v in pivoted_df.columns:
    m = re.match(r'([A-Z]+)(?:_(\d+)_TRIES)?', v)
    if m:
        base = m.group(1)
        tries = m.group(2) if m.group(2) else '-'
        variant_info.append((v, base, tries))
    else:
        variant_info.append((v, v, '-'))

# Group columns by base variant
from collections import OrderedDict
grouped = OrderedDict()
for v, base, tries in variant_info:
    grouped.setdefault(base, []).append((v, tries))

# Build header rows
header1 = ['Variant']
header2 = ['Tries']
cmidrules = []
col_idx = 2  # LaTeX columns start at 1, first is 'Variant'

for base, cols in grouped.items():
    n = len(cols)
    if n == 1:
        header1.append(base)
        header2.append('-')
        # No cmidrule needed for single columns
        col_idx += 1
    else:
        header1 += [f'\\multicolumn{{{n}}}{{c}}{{{base}}}']
        header2 += [tries for _, tries in cols]
        # cmidrule for this group
        start = col_idx
        end = col_idx + n - 1
        cmidrules.append(f'\\cmidrule(lr){{{start}-{end}}}')
        col_idx += n

header1_line = ' & '.join(header1) + r' \\'
header2_line = ' & '.join(header2) + r' \\'
cmidrules_line = '\n    '.join(cmidrules)

# --- BUILD THE TABLE ---
latex_table = r"""\begin{table}[H]
  \caption{Number of test execution failures by exception type and (generalization) variant.}
  \label{tab:exclusions-test-fails}
  \begin{tabular}{l""" + "r" * (len(pivoted_df.columns)) + r"""}
    \toprule
    """ + header1_line + "\n    " + cmidrules_line + "\n    " + header2_line + r"""
    \midrule
"""

# Data rows
for failure_type, row in pivoted_df.iterrows():
    row_str = "    " + failure_type + " & " + " & ".join(str(x) for x in row.values) + r" \\"
    latex_table += row_str + "\n"

latex_table += r"""    \bottomrule
  \end{tabular}
\end{table}
"""

print(latex_table)

## Overall processing successes and failures

In [ ]:
import pandas as pd

# Projects that do NOT successfully pass all processing steps:
# "SELECT * FROM v_project_failures" # 1150 results
# Projects that DO successfully pass all processing steps:
# "SELECT * FROM v_projects_successes" # 10 results

# Projects that show up in neither of the above two views:
# """
# SELECT id, project_name(id) FROM project
# WHERE project_name(id) NOT IN (
#     SELECT project_name FROM v_project_failures
#     UNION ALL
#     SELECT project_name FROM v_projects_successes
# )
# """
# 1 result: 6475, github_com_HdrHistogram_HdrHistogram
# Is not listed in v_project_failures because it stops at an assertion-level task,
# but the view only tracks when processing stops at a project-level task.

import pandas as pd
import re

# 1. Get the summary table
summary_query = """
SELECT NULL as step, NULL as stage, 'Total projects' AS status, (SELECT SUM(count) FROM v_project_failures_summary) + (SELECT COUNT(*) FROM v_projects_successes) AS count
UNION ALL
SELECT step, stage, stage, count FROM v_project_failures_summary
UNION ALL
SELECT NULL, NULL, 'Successfully processed', COUNT(*) FROM v_projects_successes
"""
df = pd.read_sql_query(summary_query, conn)

# 2. Get all failures with stage and info
failures_df = pd.read_sql_query("SELECT stage, info FROM v_project_failures", conn)

# 3. Define cause patterns as DataFrame for vectorized matching
cause_patterns = pd.DataFrame([
    # SETUP_PROJECT
    ('SETUP_PROJECT', r'artifacts could not be resolved', 'dependency resolution error'),
    ('SETUP_PROJECT', r'Could not find artifact', 'dependency resolution error'),
    ('SETUP_PROJECT', r'PluginVersionResolutionException', 'dependency resolution error'),
    ('SETUP_PROJECT', r'Could not resolve dependencies', 'dependency resolution error'),
    ('SETUP_PROJECT', r'Unresolveable build extension', 'dependency resolution error'),
    ('SETUP_PROJECT', r'Detected the following recursive expression cycle', 'dependency resolution error'),
    ('SETUP_PROJECT', r'must be a valid version', 'dependency resolution error'),
    ('SETUP_PROJECT', r'must specify an absolute path', 'dependency resolution error'),
    ('SETUP_PROJECT', r"Could not find goal 'build-classpath' in plugin", 'dependency resolution error'),
    ('SETUP_PROJECT', r'Error injecting:', 'dependency resolution error'),
    ('SETUP_PROJECT', r'No supported test framework identified', 'sources / tests not found'),
    ('SETUP_PROJECT', r'Test source path .+ does not exist.', 'sources / tests not found'),
    ('SETUP_PROJECT', r'Main source path .+ does not exist.', 'sources / tests not found'),
    # BUILD_PROJECT_ORIGINAL
    ('BUILD_PROJECT_ORIGINAL', r'teralizer.util.ConsoleCommandException', 'compilation error'),
    ('BUILD_PROJECT_ORIGINAL', r'Main compiled path .+ does not exist.', 'compilation outputs not found'),
    ('BUILD_PROJECT_ORIGINAL', r'Test compiled path .+ does not exist.', 'compilation outputs not found'),
    # BUILD_SPOON_MODEL
    ('BUILD_SPOON_MODEL', r'Modules are only available since Java 9.', 'Spoon execution error'),
    ('BUILD_SPOON_MODEL', r'The type package-info is already defined', 'Spoon execution error'),
    ('BUILD_SPOON_MODEL', None, 'Spoon execution error'),
    # EXECUTE_TESTS_ORIGINAL
    ('EXECUTE_TESTS_ORIGINAL', r'teralizer.util.ConsoleCommandException', 'JUnit execution error'), # @TODO: failed execution => 13 total: (i) 5x There was an error in the forked process, (ii) 4x The forked VM terminated without properly saying goodbye., (iii) 1x Exception: relation 'foorel' for 'foonoSource' does not have source, (iv) 1x Package-cycle found
    ('EXECUTE_TESTS_ORIGINAL', r'Command execution timeout exceeded.', 'timeout exceeded'),
    # COLLECT_JUNIT_REPORTS_ORIGINAL
    ('COLLECT_JUNIT_REPORTS_ORIGINAL', r'Report directory .+ does not exist.', 'JUnit outputs not found'),
    ('COLLECT_JUNIT_REPORTS_ORIGINAL', r'Test file .+ does not exist.', 'JUnit outputs not found'),
    # BUILD_PROJECT_INSTRUMENTED
    ('BUILD_PROJECT_INSTRUMENTED', r'teralizer.util.ConsoleCommandException', 'compilation error'), # @TODO: 1 total: Missing import for Matchers.any which results in /Users/joaichberger/Projects/test-generalization-dev/projects/github_com_koraktor_steam-condenser-java/src/test/java/com/github/koraktor/steamcondenser/servers/sockets/_SteamSocketTest_Instrumented_testReceiveIntoExistingBuffer_538531_Test.java:[56,32] cannot find symbol
    # EXECUTE_TESTS_INITIAL
    ('EXECUTE_TESTS_INITIAL', r'All tests of the project are excluded.', 'all tests excluded'),
    ('EXECUTE_TESTS_INITIAL', r'Command execution timeout exceeded.', 'timeout exceeded'),
    # COLLECT_JACOCO_DATA_INITIAL
    ('COLLECT_JACOCO_DATA_INITIAL', r'Report file .+ does not exist.', 'JaCoCo outputs not found'),
    ('COLLECT_JACOCO_DATA_INITIAL', r'teralizer.util.ConsoleCommandException', 'JaCoCo execution error'), # @TODO: 1 total: "Can't add different class with same name"
    # COLLECT_PIT_DATA_INITIAL
    ('COLLECT_PIT_DATA_INITIAL', r'Command execution timeout exceeded.', 'timeout exceeded'),
    ('COLLECT_PIT_DATA_INITIAL', r'All classes of the project are excluded.', 'all classes excluded'),
    ('COLLECT_PIT_DATA_INITIAL', r'Report file .+ does not exist.', 'PIT outputs not found'),
    ('COLLECT_PIT_DATA_INITIAL', r'teralizer.util.ConsoleCommandException', 'PIT execution error'), # @TODO: 16: (i) 1x Coverage generation slave exited abnormally! (ii) 6x Coverage generation minion exited abnormally! (MINION_DIED), (iii) 2x Coverage generation minion exited abnormally! (UNKNOWN_ERROR) (iv) 7x tests did not pass without mutation, (v) 1x Cannot construct org.pitest.mutationtest.MutationCoverageReport as it does not have a no-args constructor
    ('COLLECT_PIT_DATA_INITIAL', r'Failed to map coverage record to a test / generalization.', 'failed to map PIT data to a test'),
    # COLLECT_PIT_DATA_GENERALIZED
    ('COLLECT_PIT_DATA_GENERALIZED', r'All generalized tests of the project are excluded.', 'all generalizations excluded'),
    ('COLLECT_PIT_DATA_GENERALIZED', r'Failed to map coverage record to a test / generalization.', 'failed to map PIT data to a generalization'),
], columns=['stage', 'pattern', 'desc'])

# SELECT * FROM v_project_failures
# WHERE
#     stage = 'COLLECT_PIT_DATA_GENERALIZED' AND -- 266 total @TODO: 266->270
#     info NOT LIKE '%%' AND -- 269 total
#     info NOT LIKE '%%'; -- 1 total @TODO: Fix this.


# 4. Classify each failure (vectorized, if possible)
def match_cause(row):
    info = row['info']
    for _, pat in cause_patterns.iterrows():
        # Handle null pattern: match if info is null
        if pat['pattern'] is None and row['stage'] == pat['stage'] and pd.isnull(info):
            return pat['desc']
        # Handle normal regex pattern: match if info is string and pattern is not None
        if (
            row['stage'] == pat['stage']
            and pat['pattern'] is not None
            and isinstance(info, str)
            and re.search(pat['pattern'], info)
        ):
            return pat['desc']
    return 'other'

failures_df['cause_desc'] = failures_df.apply(match_cause, axis=1)

# 5. Aggregate cause counts per stage
stage_cause_counts = (
    failures_df.groupby(['stage', 'cause_desc'])
    .size()
    .reset_index(name='count')
)

# 6. Build stage -> "desc1 (n1), desc2 (n2), ..." mapping
def format_causes(df):
    return ', '.join(f"{row['cause_desc']} ({row['count']})" for _, row in df.iterrows())

stage_to_causes = (
    stage_cause_counts
    .groupby('stage')[['cause_desc', 'count']]
    .apply(format_causes)
    .to_dict()
)

# 7. Compute failures and remaining
def format_remaining(n, total):
    percent = 100 * n / total if total else 0
    percent_str = f"{percent:.1f}"
    if percent == 100:
        percent_str = "\\phantom{.}100"
    if percent < 10:
        percent_str = f"\\phantom{{0}}{percent_str}"
    return f"{n}\\; ({percent_str} \\%)"

total_projects = int(df.loc[df['status'] == 'Total projects', 'count'].values[0])
failures_per_stage = df[
    (df['status'] != 'Total projects') &
    (df['status'] != 'Successfully processed')
].copy()
failures_per_stage['Failures'] = failures_per_stage['count'].astype(int)
failures_per_stage = failures_per_stage.sort_values('step')
failures_per_stage['Remaining'] = total_projects - failures_per_stage['Failures'].cumsum()

# 8. Escape LaTeX special characters
def escape_latex(s):
    return str(s).replace('&', '\\&').replace('%', '\\%').replace('_', '\\_')

# 9. Build LaTeX tables
total_row = {
    'Processing Stage': 'Total projects',
    'Failures': '-',
    'Remaining': format_remaining(total_projects, total_projects),
}
success_count = int(df.loc[df['status'] == 'Successfully processed', 'count'].values[0])
success_row = {
    'Processing Stage': 'Successfully processed',
    'Failures': '-',
    'Remaining': format_remaining(success_count, total_projects),
}
failure_rows = []
for _, row in failures_per_stage.iterrows():
    stage = escape_latex(row['status'])
    failure_rows.append({
        'Processing Stage': stage,
        'Failures': int(row['Failures']),
        'Remaining': format_remaining(int(row['Remaining']), total_projects),
    })

latex_table_summary = r"""\begin{table}[H]
  \caption{Number of processing failures and remaining projects per processing stage.}
  \label{tab:processing-failures-per-stage}
  \begin{tabular}{l r r}
    \toprule
    Processing Stage & Failures & Remaining Projects \\
    \midrule
"""
latex_table_summary += f"    {total_row['Processing Stage']} & {total_row['Failures']} & {total_row['Remaining']} \\\\\n"
latex_table_summary += "    \\cmidrule(lr){1-3}\n"
for row in failure_rows:
    latex_table_summary += f"    {row['Processing Stage']} & {row['Failures']} & {row['Remaining']} \\\\\n"
latex_table_summary += "    \\cmidrule(lr){1-3}\n"
latex_table_summary += f"    {success_row['Processing Stage']} & {success_row['Failures']} & {success_row['Remaining']} \\\\\n"
latex_table_summary += r"""    \bottomrule
  \end{tabular}
\end{table}
"""

stage_order = (
    failures_per_stage[['stage', 'step']]
    .drop_duplicates()
    .sort_values('step')
    .set_index('stage')
    .index.tolist()
)

ordered_stage_causes = [
    (stage, stage_to_causes.get(stage, ''))
    for stage in stage_order
    if stage in stage_to_causes
]

latex_table_causes = r"""\begin{table}[H]
  \caption{Causes of processing failures per processing stage.}
  \label{tab:processing-failure-causes}
  \begin{tabularx}{\textwidth}{l X}
    \toprule
    Processing Stage & Causes of Processing Failures \\
    \midrule
"""
for stage, causes in ordered_stage_causes:
    latex_table_causes += f"    {escape_latex(stage)} & {causes} \\\\\n"
latex_table_causes += r"""    \bottomrule
  \end{tabularx}
\end{table}
"""

print(latex_table_summary)
print(latex_table_causes)

# # @TODO: Check which tests are failing due to generalization bugs vs. weak preconditions.
# "SELECT * FROM junit_test_report AS r WHERE result != 'PASSED' AND variant_name(r.stage, r.variant) != 'ORIGINAL';"
